# TurboQuant — Step-by-Step Visual Guide

**Paper:** [TurboQuant: Online Vector Quantization with Near-optimal Distortion Rate (ICLR 2026)](https://arxiv.org/abs/2504.19874)

**Goal of this notebook:**
Take *one* real vector and walk it through every step of TurboQuant.
At each step: see the math → see the code → see the picture → understand *why*.

By the end you will be able to answer:
- What does TurboQuant actually do to a vector?
- How do inner products survive compression?
- Where in *your* projects can you drop this in?

---


## 0 · The Problem: Vectors Are Expensive

Modern AI represents everything as a **dense vector** of floats.

| Thing | Typical shape | Memory (float32) |
|-------|---------------|-----------------|
| One word embedding | (768,) | 3 KB |
| One KV-cache row (GPT-4 layer) | (128,) | 512 B |
| 1 M document embeddings | (1M, 768) | **3 GB** |
| 100 B tokens × KV cache | (100B, 128) | **50 TB** |

The numbers blow up fast.
**Question:** can we compress each vector from 32 bits/dim → 2 bits/dim
without breaking the operations we care about (inner products, nearest-neighbour search)?

**TurboQuant says yes**, and proves it optimally.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from scipy.special import erfinv
from scipy.stats import norm as sp_norm

# ── style ────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "#0f1117",
    "axes.facecolor":   "#1a1d27",
    "axes.edgecolor":   "#2a2d3e",
    "axes.labelcolor":  "#e2e8f0",
    "text.color":       "#e2e8f0",
    "xtick.color":      "#64748b",
    "ytick.color":      "#64748b",
    "grid.color":       "#2a2d3e",
    "grid.linewidth":   0.6,
    "axes.grid":        True,
    "figure.dpi":       110,
    "font.size":        11,
})

C0, C1, C2, C3, C4 = "#6366f1", "#22c55e", "#f59e0b", "#ef4444", "#8b5cf6"

np.random.seed(42)
print("Imports OK.")


---
## 1 · Meet Our Vector

We'll use a **16-dimensional** vector throughout so every coordinate is visible.
Think of it as a tiny embedding — the kind your encoder produces for a sentence or an image patch.


In [ ]:
D = 16   # small enough to visualise every coordinate

# A fixed embedding — imagine this came out of a transformer encoder
x = np.array([ 0.82, -0.31,  1.23,  0.07, -0.94,  0.51, -0.19,  1.14,
                0.43, -0.73,  0.28, -1.31,  0.63, -0.08,  0.91, -0.48])

print(f"Vector x  (d={D})")
print(f"  values : {x}")
print(f"  L2 norm: {np.linalg.norm(x):.4f}")
print(f"  memory : {x.nbytes} bytes  ({x.nbytes*8} bits)")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
bars = ax.bar(range(D), x, color=[C0 if v >= 0 else C3 for v in x], alpha=0.85, width=0.7)
ax.axhline(0, color="#64748b", lw=1)
ax.set_xticks(range(D)); ax.set_xticklabels([f"d{i}" for i in range(D)], fontsize=9)
ax.set_title("Our vector x  — 16 floats, each 32 bits  =  512 bits total", pad=10)
ax.set_ylabel("Coordinate value")
for i, v in enumerate(x):
    ax.text(i, v + (0.05 if v >= 0 else -0.12), f"{v:.2f}", ha="center", fontsize=7, color="#e2e8f0")
plt.tight_layout(); plt.show()


---
## 2 · Step 1 — Normalise to the Unit Sphere

### Why?
TurboQuant's theory assumes $\|x\|_2 = 1$.
We **separate** the magnitude from the direction:

$$\hat{x} = \frac{x}{\|x\|_2}, \quad \text{store } \ell = \|x\|_2 \text{ as one fp32}$$

At reconstruction time we multiply back: $\tilde{x} = \ell \cdot \widehat{\tilde{x}}$.

**Cost:** 1 extra float32 (32 bits) per vector — negligible at $d=768$.


In [ ]:
norm_x = np.linalg.norm(x)
x_hat  = x / norm_x               # unit-norm vector

print(f"‖x‖₂     = {norm_x:.4f}  ← stored as one fp32")
print(f"‖x̂‖₂    = {np.linalg.norm(x_hat):.8f}  ← exactly 1 (unit sphere)")
print(f"\nNormalised x̂:\n{x_hat.round(4)}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

for ax, vec, title, col in zip(
        axes,
        [x,     x_hat],
        ["x  (original, ‖x‖=%.2f)" % norm_x,
         "x̂  (normalised, ‖x̂‖=1.00)"],
        [C3, C0]):
    ax.bar(range(D), vec, color=col, alpha=0.8, width=0.7)
    ax.axhline(0, color="#64748b", lw=1)
    ax.axhline( 1, color="#22c55e", lw=0.8, ls="--", alpha=0.5, label="±1")
    ax.axhline(-1, color="#22c55e", lw=0.8, ls="--", alpha=0.5)
    ax.set_ylim(-1.6, 1.6)
    ax.set_title(title); ax.set_xticks(range(D))
    ax.set_xticklabels([f"d{i}" for i in range(D)], fontsize=8)

axes[0].set_ylabel("Value")
plt.suptitle("Normalisation: all coordinates now live in [-1, 1]", y=1.02)
plt.tight_layout(); plt.show()
print("Key property: ‖x̂‖₂ = 1  →  inner product ⟨q, x̂⟩ = cos(angle between q and x̂)")


---
## 3 · Step 2 — Random Orthogonal Rotation  ← The Key Innovation

### Why rotate at all?

Look at $\hat{x}$: some coordinates are large ($|d_2|=1.23$), others near zero ($|d_3|=0.07$).
If we just quantize naively, we'd **waste bits** on small dims and **overflow** on large ones.

**Big idea:** apply an orthogonal matrix $\Pi$ so every coordinate has the *same* scale.

### How to generate $\Pi$

Draw a random $d \times d$ Gaussian matrix $M$, then do **QR decomposition**:

$$M = QR \quad \Rightarrow \quad \Pi = Q \quad (\Pi\Pi^\top = I)$$

$\Pi$ is uniformly random over all orthogonal matrices (Haar measure).

### What it guarantees (mathematically)

If $\|\hat{x}\|_2 = 1$, then each coordinate of $y = \Pi\hat{x}$ follows

$$y_j \sim \text{Beta}\!\left(\tfrac{d-1}{2},\tfrac{d-1}{2}\right) \text{ on } [-1/\sqrt{d},\, 1/\sqrt{d}]$$

which for large $d$ is very close to $\mathcal{N}\!\left(0,\,\tfrac{1}{d}\right)$.
**Every dimension gets the same distribution → optimal equal bit allocation.**


In [ ]:
# Generate rotation matrix Pi via QR decomposition
M  = np.random.randn(D, D)
Pi, _ = np.linalg.qr(M)        # Pi is orthogonal

# Rotate
y = Pi @ x_hat                  # y = Pi * x_hat

print("Rotation matrix Pi:")
print(f"  shape : {Pi.shape}")
print(f"  Pi @ Pi.T ≈ I? max error: {np.max(np.abs(Pi @ Pi.T - np.eye(D))):.2e}")
print(f"\ny = Pi @ x̂")
print(f"  values: {y.round(4)}")
print(f"  ‖y‖₂  = {np.linalg.norm(y):.8f}  (same as ‖x̂‖₂ — rotation preserves norm!)")
print(f"\nPer-coordinate std before rotation: {x_hat.std():.4f}")
print(f"Per-coordinate std after  rotation: {y.std():.4f}  ≈ 1/√d = {1/np.sqrt(D):.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))

# Left: before vs after — bar chart
ax = axes[0]
width = 0.38
ax.bar(np.arange(D) - width/2, x_hat, width, color=C3, alpha=0.8, label="x̂  (before rotation)")
ax.bar(np.arange(D) + width/2, y,     width, color=C0, alpha=0.8, label="y = Πx̂  (after rotation)")
ax.axhline(0, color="#64748b", lw=1)
ax.axhline( 1/np.sqrt(D), color=C1, lw=1, ls="--", alpha=0.7, label=f"±1/√d = ±{1/np.sqrt(D):.3f}")
ax.axhline(-1/np.sqrt(D), color=C1, lw=1, ls="--", alpha=0.7)
ax.set_title("Coordinates before vs after rotation")
ax.set_xticks(range(D)); ax.set_xticklabels([f"d{i}" for i in range(D)], fontsize=8)
ax.set_ylabel("Value"); ax.legend(fontsize=9, loc="upper right")

# Right: histogram — does y fit N(0, 1/d)?
ax = axes[1]
# Use many random unit vectors to build a proper histogram
N_stat = 2000
X_many = np.random.randn(N_stat, D)
X_many = X_many / np.linalg.norm(X_many, axis=1, keepdims=True)
Y_many = X_many @ Pi.T
coords  = Y_many.flatten()

ax.hist(coords, bins=60, density=True, color=C0, alpha=0.75, label="Empirical (2000 vectors × 16 dims)")
xg = np.linspace(-4/np.sqrt(D), 4/np.sqrt(D), 300)
ax.plot(xg, sp_norm.pdf(xg, 0, 1/np.sqrt(D)), color="white", lw=2.5,
        label=f"N(0, 1/d) = N(0, {1/D:.4f})")
ax.set_title("After rotation: coordinate distribution fits N(0, 1/d)")
ax.set_xlabel("Coordinate value"); ax.set_ylabel("Density"); ax.legend(fontsize=9)

plt.suptitle("Random rotation equalises per-dimension energy", y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
# Critical property: rotation PRESERVES inner products
q = np.random.randn(D); q /= np.linalg.norm(q)  # a random query vector
q_rot = Pi @ q

ip_original = q @ x_hat
ip_rotated  = q_rot @ y     # <Pi q, Pi x_hat>

print("Inner product preservation (rotation is isometry):")
print(f"  ⟨q, x̂⟩           = {ip_original:.6f}")
print(f"  ⟨Πq, Πx̂⟩ = ⟨Πq, y⟩ = {ip_rotated:.6f}")
print(f"  Difference          = {abs(ip_original - ip_rotated):.2e}  ← machine epsilon")
print()
print("Mathematical reason: ⟨Πa, Πb⟩ = (Πa)ᵀ(Πb) = aᵀΠᵀΠb = aᵀIb = ⟨a, b⟩")
print("Rotations never change angles — only coordinate representations change.")


---
## 4 · Step 3 — Lloyd-Max Scalar Quantisation

Now each coordinate $y_j \sim \mathcal{N}(0, 1/d)$.
We assign each $y_j$ to one of $2^b$ **centroids** $\{c_1, \ldots, c_{2^b}\}$.

### Lloyd-Max centroids

For a Gaussian source, the optimal centroids minimise the expected squared error:

$$\min_{c_1,\ldots,c_{2^b}} \;\mathbb{E}\!\left[(y - c_{k^*})^2\right], \quad k^* = \arg\min_k |y - c_k|$$

For $\mathcal{N}(0,1)$, well-known values (scale by $1/\sqrt{d}$ for our case):

| Bits | Levels | Centroids (×$1/\sqrt{d}$) |
|------|--------|---------------------------|
| 1    | 2      | ±0.7979 |
| 2    | 4      | ±0.4528, ±1.5104 |
| 3    | 8      | ±0.2451, ±0.7560, ±1.3439, ±2.1520 |

**Storage:** instead of a 32-bit float, we store a $b$-bit integer index.
$\Rightarrow$ 16× compression at 2-bit vs float32.


In [ ]:
def lloyd_max_centroids(n_levels, n_samples=200_000, n_iter=300):
    # Lloyd-Max optimal centroids for N(0,1) — scale by 1/sqrt(d) at call site
    samples = np.random.randn(n_samples)
    q = np.linspace(1/(2*n_levels), 1 - 1/(2*n_levels), n_levels)
    c = np.sqrt(2) * erfinv(2*q - 1)                   # initialise at quantile midpoints
    for _ in range(n_iter):
        assign = np.argmin(np.abs(samples[:, None] - c[None, :]), axis=1)
        new_c  = np.array([samples[assign==k].mean() if (assign==k).any() else c[k]
                           for k in range(n_levels)])
        if np.max(np.abs(new_c - c)) < 1e-10: break
        c = new_c
    return np.sort(c)

# Precompute for 1–4 bits
centroids_n01 = {b: lloyd_max_centroids(2**b) for b in [1,2,3,4]}

print("Lloyd-Max centroids for N(0,1):")
for b, c in centroids_n01.items():
    print(f"  {b}-bit ({2**b} levels): {c.round(4)}")


In [ ]:
# Visualise centroids on a number line for 1-bit, 2-bit, 4-bit
fig, axes = plt.subplots(3, 1, figsize=(12, 5))
scale = 1 / np.sqrt(D)

for ax, b, col in zip(axes, [1, 2, 4], [C3, C0, C1]):
    c = centroids_n01[b] * scale                   # scale to N(0, 1/d)
    # decision boundaries = midpoints between consecutive centroids
    bounds = (c[:-1] + c[1:]) / 2

    # number line
    xmin, xmax = -4*scale, 4*scale
    ax.hlines(0, xmin, xmax, colors="#64748b", lw=2)
    ax.scatter(c, np.zeros_like(c), s=200, color=col, zorder=5,
               label=f"{b}-bit centroids", marker="D")
    for bnd in bounds:
        ax.axvline(bnd, color="#64748b", lw=1, ls=":", alpha=0.6)
    for i, ci in enumerate(c):
        ax.text(ci, 0.3, f"c{i}\n{ci:.3f}", ha="center", fontsize=9 if b<=2 else 7, color=col)
    # mark our y_j's on the number line
    ax.scatter(y, np.zeros(D), s=40, color="white", alpha=0.6, zorder=4, label="y_j values")
    ax.set_xlim(xmin, xmax); ax.set_ylim(-0.5, 0.8)
    ax.set_yticks([]); ax.set_xlabel("Coordinate value")
    ax.set_title(f"{b}-bit:  {2**b} centroids  →  {b} bits/coordinate  →  {32//b}× compression vs float32")
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(False); ax.axhline(0, color="#64748b", lw=0)

plt.suptitle("Lloyd-Max centroids: the 'snap points' for each bit-width", y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# Now quantize y to 2-bit
BITS = 2
c2 = centroids_n01[BITS] * (1/np.sqrt(D))

diffs   = np.abs(y[:, None] - c2[None, :])         # (D, 4)
indices = np.argmin(diffs, axis=1)                  # integer index per coordinate
y_hat   = c2[indices]                               # reconstructed rotated vector

print(f"2-bit quantisation of y:")
print(f"{'dim':>4}  {'y_j':>8}  {'idx':>4}  {'centroid':>10}  {'error':>10}")
for i in range(D):
    print(f"  d{i:<2}  {y[i]:+.4f}  [{indices[i]}]  {y_hat[i]:+.6f}  {y[i]-y_hat[i]:+.6f}")
print(f"\nQuantisation MSE in rotated space: {np.mean((y - y_hat)**2):.6f}")
print(f"TurboQuant upper bound (theory):  {np.sqrt(3*np.pi)/2/4**BITS:.6f}")
print(f"Information-theoretic lower bound: {1/4**BITS:.6f}")


---
## 5 · Step 4 — Dequantise (Rotate Back + Rescale)

We have $\hat{y}$ (quantised rotated vector). To reconstruct $\tilde{x}$:

$$\tilde{x} = \ell \cdot \Pi^\top \hat{y}$$

Because $\Pi$ is orthogonal, $\Pi^\top = \Pi^{-1}$  — no matrix inversion needed, just a transpose.


In [ ]:
x_hat_recon = Pi.T @ y_hat      # rotate back (Pi^T = Pi^{-1})
x_recon     = norm_x * x_hat_recon   # rescale by stored norm

print("Reconstruction quality:")
print(f"  Original  x:  {x.round(4)}")
print(f"  Recovered x̃:  {x_recon.round(4)}")
print(f"  L2 error ‖x - x̃‖₂:   {np.linalg.norm(x - x_recon):.6f}")
print(f"  Relative error:         {np.linalg.norm(x - x_recon)/np.linalg.norm(x)*100:.2f}%")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: original vs reconstructed
ax = axes[0]
width = 0.38
ax.bar(np.arange(D) - width/2, x,       width, color=C3, alpha=0.85, label="Original x")
ax.bar(np.arange(D) + width/2, x_recon, width, color=C0, alpha=0.85, label="Reconstructed x̃ (2-bit)")
ax.axhline(0, color="#64748b", lw=1)
ax.set_xticks(range(D)); ax.set_xticklabels([f"d{i}" for i in range(D)], fontsize=8)
ax.set_ylabel("Value"); ax.legend()
ax.set_title("Original vs Reconstructed — hard to tell apart!")

# Right: per-coordinate error
ax = axes[1]
errors = x - x_recon
ax.bar(range(D), errors, color=[C2 if e > 0 else C4 for e in errors], alpha=0.85)
ax.axhline(0, color="#64748b", lw=1)
ax.set_xticks(range(D)); ax.set_xticklabels([f"d{i}" for i in range(D)], fontsize=8)
ax.set_title(f"Reconstruction error per coordinate  (MSE = {np.mean(errors**2):.5f})")
ax.set_ylabel("x_j - x̃_j")

plt.suptitle(f"2-bit TurboQuant-MSE: {D*32} bits → {D*BITS+32} bits  ({D*32/(D*BITS+32):.1f}× compression)", y=1.02)
plt.tight_layout(); plt.show()


---
## 6 · The Inner-Product Problem

### Why MSE-optimal ≠ inner-product-optimal

Vector search and attention both need **inner products**:
$$\text{score}(q, x) = \langle q, x \rangle$$

Does $\langle q, \tilde{x}_{\text{mse}} \rangle \approx \langle q, x \rangle$?

**Sort of, but with bias.**
At 1-bit, the MSE centroid is $\pm\sqrt{2/\pi d}$ ≈ $0.798/\sqrt{d}$.
But the true $\mathbb{E}[y_j] = 0$ and $\mathbb{E}[\hat{y}_j \,|\, y_j] \neq y_j$ in general.
This creates a **multiplicative shrinkage bias**: estimated inner products are systematically smaller.

Below we see this empirically.


In [ ]:
N_test = 500
X_test = np.random.randn(N_test, D)
X_test = X_test / np.linalg.norm(X_test, axis=1, keepdims=True)
Q_test = np.random.randn(80, D)
Q_test = Q_test / np.linalg.norm(Q_test, axis=1, keepdims=True)

# True inner products
true_ip = (Q_test @ X_test.T).flatten()                    # (80 * 500,)

# MSE reconstruction for each bit-width
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, b, col in zip(axes, [1, 2, 4], [C3, C0, C1]):
    c = centroids_n01[b] * (1/np.sqrt(D))
    # quantise all X_test
    Y_test  = X_test @ Pi.T
    idx_t   = np.argmin(np.abs(Y_test[:, :, None] - c[None, None, :]), axis=2)
    Yh_test = c[idx_t]
    Xh_test = Yh_test @ Pi                                  # rotate back (no rescale, already unit)

    est_ip  = (Q_test @ Xh_test.T).flatten()
    bias    = np.mean(est_ip) - np.mean(true_ip)

    lim = max(np.abs(true_ip).max(), np.abs(est_ip).max()) * 1.1
    ax.scatter(true_ip[::5], est_ip[::5], s=4, alpha=0.4, color=col, rasterized=True)
    ax.plot([-lim, lim], [-lim, lim], "--", color="#64748b", lw=1.5, label="y = x (perfect)")
    ax.set_title(f"{b}-bit MSE  (bias = {bias:+.4f})", fontsize=10)
    ax.set_xlabel("True ⟨q, x⟩"); ax.set_ylabel("Estimated"); ax.legend(fontsize=8)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)

plt.suptitle("MSE variant: inner product estimates are biased (points below y = x line)", y=1.02)
plt.tight_layout(); plt.show()

print("At 1-bit, the MSE reconstruction systematically under-estimates inner products.")
print("This biases ranking in nearest-neighbour search → wrong results!")


---
## 7 · Step 5 — Residual QJL (Fix the Bias)

### The elegant fix

After MSE quantisation, the **residual** is:
$$r = \hat{x} - \tilde{x}_{\text{mse}}$$

$r$ carries the information lost in Step 3. We can't store $r$ (it's a full float vector),
but we can apply a **1-bit Quantised Johnson-Lindenstrauss (QJL)** transform:

$$\text{qjl} = \text{sign}(S \cdot r) \in \{-1, +1\}^d$$

where $S \in \mathbb{R}^{d \times d}$ is a fixed random Gaussian matrix.

### Why does 1-bit QJL fix the bias?

The key theorem (Lemma 4 in the paper):

$$\mathbb{E}\!\left[\left\langle q,\; \frac{\sqrt{\pi/2}}{d}\, \gamma \cdot S^\top \text{qjl} \right\rangle\right] = \langle q,\, r \rangle$$

where $\gamma = \|r\|_2$. In plain English:
the QJL reconstruction is an **unbiased estimator** of the residual's inner product with any query.

### Full prod reconstruction

$$\tilde{x}_{\text{prod}} = \tilde{x}_{\text{mse}} + \underbrace{\frac{\sqrt{\pi/2}}{d}\, \gamma \cdot S^\top \cdot \text{sign}(Sr)}_{\text{QJL correction}}$$

$$\mathbb{E}[\langle q, \tilde{x}_{\text{prod}} \rangle] = \langle q, \tilde{x}_{\text{mse}} \rangle + \langle q, r \rangle = \langle q, \hat{x} \rangle = \langle q, x \rangle / \ell$$


In [ ]:
# Fixed S matrix (same as TurboQuant class)
rng_s = np.random.RandomState(42)
S = rng_s.randn(D, D)

BITS_PROD = 2      # total bits; uses (BITS_PROD - 1) for MSE + 1 for QJL
b_mse     = BITS_PROD - 1

# ── Step 5a: (b-1)-bit MSE on x_hat ─────────────────────────────────────────
c1bit     = centroids_n01[b_mse] * (1/np.sqrt(D))
y_1bit    = Pi @ x_hat
idx_1bit  = np.argmin(np.abs(y_1bit[:, None] - c1bit[None, :]), axis=1)
yh_1bit   = c1bit[idx_1bit]
xh_mse    = Pi.T @ yh_1bit                         # 1-bit MSE reconstruction (unit norm)

# ── Step 5b: residual ────────────────────────────────────────────────────────
r      = x_hat - xh_mse                            # residual (still full-float)
gamma  = np.linalg.norm(r)                         # stored as one fp32

# ── Step 5c: 1-bit QJL on residual ──────────────────────────────────────────
qjl_bits = np.sign(S @ r)                          # {-1, +1}^d  ← 1 bit each
qjl_bits = np.where(qjl_bits == 0, 1.0, qjl_bits)

# ── Step 5d: reconstruct correction ─────────────────────────────────────────
correction = (np.sqrt(np.pi / 2) / D) * gamma * (S.T @ qjl_bits)

# ── Step 5e: final reconstruction ───────────────────────────────────────────
xh_prod = xh_mse + correction

print("Prod-variant reconstruction for our vector x:")
print(f"  ‖r‖₂ (residual norm, stored as fp32) = {gamma:.6f}")
print(f"  QJL bits (first 8):  {qjl_bits[:8].astype(int)}")
print(f"\n  ‖x̂ - x̂_mse‖₂  = {np.linalg.norm(x_hat - xh_mse):.6f}  (before QJL fix)")
print(f"  ‖x̂ - x̂_prod‖₂ = {np.linalg.norm(x_hat - xh_prod):.6f}  (after QJL fix)")


In [ ]:
# Show the residual and the QJL correction
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))

ax = axes[0]
ax.bar(range(D), r, color=[C2 if v >= 0 else C4 for v in r], alpha=0.85)
ax.axhline(0, color="#64748b", lw=1)
ax.set_title(f"Residual  r = x̂ - x̂_mse  (‖r‖₂ = {gamma:.4f})")
ax.set_xticks(range(D)); ax.set_xticklabels([f"d{i}" for i in range(D)], fontsize=8)
ax.set_ylabel("r_j")

ax = axes[1]
ax.bar(range(D), qjl_bits, color=[C0 if v > 0 else C3 for v in qjl_bits], alpha=0.85, width=0.7)
ax.axhline(0, color="#64748b", lw=1)
ax.set_yticks([-1, 0, 1]); ax.set_yticklabels(["-1", "0", "+1"])
ax.set_title(f"QJL bits = sign(S·r)  ← 1 bit per coord  ({D} bits total)")
ax.set_xticks(range(D)); ax.set_xticklabels([f"d{i}" for i in range(D)], fontsize=8)
ax.set_ylabel("sign bit")

ax = axes[2]
width = 0.27
ax.bar(np.arange(D) - width,   x_hat,   width, color="#64748b", alpha=0.6, label="x̂  (true)")
ax.bar(np.arange(D),           xh_mse,  width, color=C3, alpha=0.85, label="x̂_mse  (1-bit MSE)")
ax.bar(np.arange(D) + width,   xh_prod, width, color=C0, alpha=0.85, label="x̂_prod  (+QJL fix)")
ax.axhline(0, color="#64748b", lw=1)
ax.set_title("QJL correction pulls reconstruction closer to truth")
ax.set_xticks(range(D)); ax.set_xticklabels([f"d{i}" for i in range(D)], fontsize=8)
ax.legend(fontsize=9)

plt.suptitle("Step 5: QJL residual correction fixes the inner-product bias", y=1.02)
plt.tight_layout(); plt.show()


---
## 8 · Proving Unbiasedness: Inner Product Comparison

Let's verify on many random queries that TurboQuant-prod gives **unbiased** inner products.


In [ ]:
N_large = 1000
X_large = np.random.randn(N_large, D)
X_large /= np.linalg.norm(X_large, axis=1, keepdims=True)

Q_large = np.random.randn(50, D)
Q_large /= np.linalg.norm(Q_large, axis=1, keepdims=True)

true_ips = (Q_large @ X_large.T).flatten()

# MSE (2-bit)
c2b = centroids_n01[2] * (1/np.sqrt(D))
Y_l = X_large @ Pi.T
idx_l = np.argmin(np.abs(Y_l[:, :, None] - c2b[None, None, :]), axis=2)
Yh_l  = c2b[idx_l]
Xh_mse_l = Yh_l @ Pi
est_mse_ips = (Q_large @ Xh_mse_l.T).flatten()

# Prod (2-bit = 1-bit MSE + 1-bit QJL)
c1b = centroids_n01[1] * (1/np.sqrt(D))
Y_l1 = X_large @ Pi.T
idx_l1 = np.argmin(np.abs(Y_l1[:, :, None] - c1b[None, None, :]), axis=2)
Yh_l1 = c1b[idx_l1]
Xh_mse_l1 = Yh_l1 @ Pi                                    # (1-bit) MSE reconstr.
R_l = X_large - Xh_mse_l1                                  # residuals (N, D)
G_l = np.linalg.norm(R_l, axis=1, keepdims=True)           # (N, 1) gammas
qjl_l = np.sign(R_l @ S.T)                                 # (N, D)
qjl_l = np.where(qjl_l == 0, 1.0, qjl_l)
corr_l = (np.sqrt(np.pi/2)/D) * G_l * (qjl_l @ S)
Xh_prod_l = Xh_mse_l1 + corr_l
est_prod_ips = (Q_large @ Xh_prod_l.T).flatten()

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
lim = np.abs(true_ips).max() * 1.05

for ax, est, label, col in zip(
    axes,
    [est_mse_ips, est_prod_ips],
    ["2-bit MSE  (biased)", "2-bit Prod  (unbiased, 1-bit MSE + 1-bit QJL)"],
    [C3, C0]
):
    ax.scatter(true_ips[::3], est[::3], s=5, alpha=0.4, color=col, rasterized=True)
    ax.plot([-lim, lim], [-lim, lim], "--", color="white", lw=1.5, alpha=0.6, label="Perfect y = x")

    bias = np.mean(est - true_ips)
    var  = np.var(est - true_ips)
    ax.set_title(f"{label}\nbias = {bias:+.5f}   variance = {var:.5f}", fontsize=10)
    ax.set_xlabel("True ⟨q, x⟩"); ax.set_ylabel("Estimated")
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.legend(fontsize=9)
    ax.text(0.05, 0.95, f"bias={bias:+.4f}\nvar={var:.4f}",
            transform=ax.transAxes, va="top", fontsize=10,
            color=col, bbox=dict(fc="#0f1117", ec=col, alpha=0.8, pad=4))

plt.suptitle("MSE variant: biased   |   Prod variant: unbiased inner products", y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

print(f"MSE  variant bias: {np.mean(est_mse_ips  - true_ips):+.5f}")
print(f"Prod variant bias: {np.mean(est_prod_ips - true_ips):+.5f}  ← near zero ✓")


---
## 9 · The Full Pipeline at a Glance

```
INPUT x  (d × float32 = d×32 bits)
  │
  ├─ store ℓ = ‖x‖₂  (1 × fp32 = 32 bits)
  │
  ▼  x̂ = x/ℓ

ROTATE  y = Π x̂         ← one matrix multiply, O(d²)
  │                        Π generated once, shared across all vectors
  ▼

QUANTISE  idx = argmin_k |y_j - c_k|   ← per coordinate, O(d × 2^b)
  │        store idx  (d × b bits)
  │
  ├──── MSE VARIANT DONE ───────────────────────────────────────────────┐
  │     Decompress: ŷ = c[idx],  x̃ = ℓ · ΠᵀŷTuples                   │
  │                                                                      │
  ▼  (prod variant only)                                                 │

RESIDUAL  r = x̂ - Π†ŷ                                                  │
  │                                                                      │
  ├─ store γ = ‖r‖₂  (1 × fp32 = 32 bits)                              │
  │                                                                      │
  ▼                                                                      │

QJL  qjl = sign(S·r)  ∈ {-1,+1}^d   ← 1 bit per coord               │
  │   store qjl  (d × 1 bit)                                            │
  │                                                                      │
  ▼  (prod variant done)                                                 │
                                                                         │
OUTPUT (prod):  (idx, qjl, γ, ℓ)   total = d·b + d·1 + 32 + 32 bits  │
OUTPUT (mse):   (idx, ℓ)            total = d·b + 32 bits ◄────────────┘
```

### Compression ratio (d=768, 2-bit prod)
- Original: $768 \times 32 = 24{,}576$ bits
- Compressed: $768 \times 2 + 768 \times 1 + 32 + 32 = 2{,}368$ bits
- **Ratio: ~10×**  (and inner products are unbiased)


In [ ]:
def compression_summary(d, b, variant="prod"):
    original   = d * 32
    idx_bits   = d * b
    norm_bits  = 32
    qjl_bits   = d * 1 if variant == "prod" else 0
    gamma_bits = 32   if variant == "prod" else 0
    total      = idx_bits + norm_bits + qjl_bits + gamma_bits
    return original, total, original / total

print("Compression summary:")
print(f"{'d':>6}  {'bits':>4}  {'variant':>6}  {'original':>10}  {'compressed':>12}  {'ratio':>8}")
print("-" * 60)
for d in [16, 256, 768, 4096]:
    for b in [1, 2, 4]:
        for var in ["mse", "prod"]:
            orig, comp, ratio = compression_summary(d, b, var)
            print(f"  d={d:<5}  {b}-bit  {var:>4}:  {orig:>8} bits  {comp:>10} bits  {ratio:>6.1f}×")
    print()


---
## 10 · Use Cases — Where Can You Drop TurboQuant In?

For each use case:
1. What does the **original** computation look like?
2. How does **TurboQuant** change the storage/computation?
3. Why do results stay the same?


### 10.1 · Vector Search / Semantic Search

**What it is:** find the top-k most similar items to a query from a database of N embeddings.

**Original computation:**
```python
scores = query @ database.T          # (N,) dot products — O(N·d) FLOPs
top_k  = np.argsort(-scores)[:k]
```

**With TurboQuant:**
```python
# Offline (once):
idx, norms = tq.quantize(database)                     # compress: N × (b·d + 32) bits
db_hat = tq.dequantize(idx, norms)                    # or keep compressed, expand on demand

# Online (per query):
scores_approx = query @ db_hat.T                      # same dot product, same O(N·d) FLOPs
top_k_approx  = np.argsort(-scores_approx)[:k]        # same ranking code
```

**Why same results:** TurboQuant-prod gives $\mathbb{E}[\langle q, \tilde{x}\rangle] = \langle q, x\rangle$.
Ranking by unbiased estimates = same top-k with high probability.

**Your project mapping:**
→ `ai-news-aggregator`: article embedding index
→ Any RAG pipeline: document chunk retrieval
→ Any recommender: item embedding lookup


In [ ]:
# DEMO: Vector search quality
N_db   = 500
DB     = np.random.randn(N_db, D);  DB  /= np.linalg.norm(DB,  axis=1, keepdims=True)
query  = np.random.randn(D);        query /= np.linalg.norm(query)

# Ground truth top-10
true_scores  = query @ DB.T
true_top10   = set(np.argsort(-true_scores)[:10])

print(f"{'Bits':>5}  {'Variant':>6}  {'Recall@10':>10}  {'Compression':>12}")
print("-" * 40)
for b in [1, 2, 3, 4]:
    for var in ["mse", "prod"]:
        if var == "prod" and b < 2:
            continue        # prod needs b>=2 (1 bit MSE + 1 bit QJL)
        b_mse = b if var == "mse" else b - 1
        cb    = centroids_n01[b_mse] * (1/np.sqrt(D))
        Y_db  = DB @ Pi.T
        idx_d = np.argmin(np.abs(Y_db[:, :, None] - cb[None, None, :]), axis=2)
        Yh_d  = cb[idx_d]
        Xh_d  = Yh_d @ Pi

        if var == "prod" and b >= 2:
            R_d  = DB - Xh_d
            G_d  = np.linalg.norm(R_d, axis=1, keepdims=True)
            q_d  = np.sign(R_d @ S.T); q_d = np.where(q_d==0, 1.0, q_d)
            Xh_d = Xh_d + (np.sqrt(np.pi/2)/D) * G_d * (q_d @ S)

        approx_scores = query @ Xh_d.T
        approx_top10  = set(np.argsort(-approx_scores)[:10])
        recall        = len(true_top10 & approx_top10) / 10
        _, comp, ratio = compression_summary(D, b if var=="mse" else b, var)
        print(f"  {b}-bit  {var:>6}  {recall*100:>8.0f}%    {ratio:>8.1f}×")
    if b == 1: print()


### 10.2 · KV Cache Compression (Transformer Inference)

**What it is:** during autoregressive generation, the model stores Key and Value tensors
for every token in every layer. At 100K context, this dominates GPU memory.

**Original computation:**
```python
# For each attention head:
logits  = (Q @ K.T) / sqrt(d_head)        # (n_q, n_k) attention logits
weights = softmax(logits, dim=-1)          # attention pattern
output  = weights @ V                      # (n_q, d_head) attended values
```

**With TurboQuant on K:**
```python
# Store:
K_compressed = tq.quantize_prod(K)        # ~10× less GPU memory per layer

# At inference:
K_hat   = tq.dequantize_prod(*K_compressed)
logits  = (Q @ K_hat.T) / sqrt(d_head)   # same formula, same code
```

**Why same results:**
$\mathbb{E}[\langle q_i, k_j^{\text{compressed}} \rangle] = \langle q_i, k_j \rangle$
⇒ $\mathbb{E}[\text{logits}] = \text{true logits}$ ⇒ attention pattern preserved in expectation.

**Your project mapping:**
→ Any LLM serving system, any long-context inference
→ Reduces time-to-first-token and GPU memory by ~10× for KV cache


In [ ]:
# DEMO: KV cache attention fidelity
n_q, n_k, d_h = 16, 32, D

Q_attn = np.random.randn(n_q, d_h)
K_attn = np.random.randn(n_k, d_h)
K_norms = np.linalg.norm(K_attn, axis=1)
K_unit  = K_attn / K_norms[:, None]

# True attention pattern
true_logits = (Q_attn @ K_attn.T) / np.sqrt(d_h)
true_attn   = np.exp(true_logits - true_logits.max(1, keepdims=True))
true_attn  /= true_attn.sum(1, keepdims=True)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
im = axes[0].imshow(true_attn, aspect="auto", cmap="magma", vmin=0, vmax=0.15)
axes[0].set_title("True attention weights\n(ground truth)")
axes[0].set_xlabel("Key position"); axes[0].set_ylabel("Query position")

for ax, b, col in zip(axes[1:], [1, 2, 4], [C3, C0, C1]):
    cb   = centroids_n01[b] * (1/np.sqrt(d_h))
    Pi_h, _ = np.linalg.qr(np.random.RandomState(42).randn(d_h, d_h))
    Y_k  = K_unit @ Pi_h.T
    idx_k = np.argmin(np.abs(Y_k[:, :, None] - cb[None, None, :]), axis=2)
    Yh_k = cb[idx_k]
    Kh   = (Yh_k @ Pi_h) * K_norms[:, None]     # rescale back

    q_logits = (Q_attn @ Kh.T) / np.sqrt(d_h)
    q_attn   = np.exp(q_logits - q_logits.max(1, keepdims=True))
    q_attn  /= q_attn.sum(1, keepdims=True)

    corr = np.corrcoef(true_attn.flatten(), q_attn.flatten())[0, 1]
    ax.imshow(q_attn, aspect="auto", cmap="magma", vmin=0, vmax=0.15)
    ax.set_title(f"{b}-bit compressed K\ncorr = {corr:.4f}")
    ax.set_xlabel("Key position")

plt.suptitle("KV Cache: attention patterns survive TurboQuant compression", y=1.03)
plt.tight_layout(); plt.show()


### 10.3 · Recommender Systems — Item Embeddings

**What it is:** score items for a user via dot product between user and item embeddings.

**Original:**  $\text{score}(u, i) = \langle e_u, e_i \rangle$

**With TurboQuant:**
- Compress all $N_{\text{item}}$ item embeddings at index build time
- At serving: $\text{score}(u, i) = \langle e_u, \tilde{e}_i \rangle$

**Why same results:** same unbiasedness argument.
Ranking of items by score is preserved.

**Scale benefit:**
100M items × 768-dim float32 = **300 GB**
100M items × 768-dim 2-bit TurboQuant ≈ **29 GB** → fits on one A100.

### 10.4 · RAG (Retrieval-Augmented Generation)

Same as Vector Search (§10.1) but embedded in an LLM pipeline:

```
User query
  → embed(query)              # dense vector
  → ANN search in TurboQuant-compressed doc index
  → retrieve top-k chunks
  → stuff into LLM context
  → generate answer
```

TurboQuant compresses the document index.
Retrieval quality is preserved (recall@10 stays high even at 2-bit).

### 10.5 · Dense Passage Retrieval / FAISS

FAISS currently uses **Product Quantisation (PQ)**, which splits dimensions into sub-vectors.
TurboQuant is a **theoretically superior alternative** to PQ:
- PQ: no distortion-rate guarantee
- TurboQuant: proven within factor √(3π)/2 ≈ 2.7 of information-theoretic optimum


---
## 11 · Decision Guide: Is TurboQuant Right For My Project?

```
Do you store or transmit dense float vectors?
│
├── NO  → TurboQuant won't help
│
└── YES
    │
    ├── What operation matters most?
    │    │
    │    ├── Minimise reconstruction error (x̃ ≈ x)
    │    │    → Use TurboQuant-MSE  (quantize / dequantize)
    │    │    → Good for: storing model weights, feature compression
    │    │
    │    └── Preserve inner products / rankings
    │         → Use TurboQuant-Prod  (quantize_prod / dequantize_prod)
    │         → Good for: vector search, KV cache, recommenders, RAG
    │
    ├── How many vectors?
    │    ├── < 10K  → probably fine without compression
    │    └── > 100K → TurboQuant gives meaningful memory savings
    │
    └── What bit-width?
         ├── 1-bit: ~32× compression, recall@10 ~70%  (aggressive)
         ├── 2-bit: ~10× compression, recall@10 ~95%  ← SWEET SPOT
         ├── 3-bit:  ~7× compression, recall@10 ~99%
         └── 4-bit:  ~6× compression, recall@10 ~99.9%
```

### Quick integration checklist

```python
from turbo_quant import TurboQuant

# 1. Init once per embedding dimension
tq = TurboQuant(d=YOUR_EMBEDDING_DIM, bits=2)

# 2. At index build time (offline)
idx, norms = tq.quantize(your_vectors)              # MSE
# OR
idx, qjl, gamma, norms = tq.quantize_prod(your_vectors)   # Prod

# 3. Store (idx, norms) or (idx, qjl, gamma, norms)

# 4. At query time (online) — same dot product, no code change
candidates = tq.dequantize(idx, norms)
scores = query_vector @ candidates.T
```


---
## 12 · Final Summary: What TurboQuant Does to a Vector

| Step | Operation | Input | Output | Purpose |
|------|-----------|-------|--------|---------|
| 1 | Normalise | x (any norm) | x̂ (unit sphere) + ℓ | Control quantisation range |
| 2 | Rotate | x̂ ∈ ℝᵈ | y = Πx̂ ∈ ℝᵈ | Equalise per-dim energy |
| 3 | Quantise | y_j ∈ ℝ | idx_j ∈ {0…2ᵇ⁻¹} | Compress 32 bits → b bits |
| 4 | Unrotate | ŷ (centroids) | x̂_mse = Πᵀŷ | Reconstruct unit vector |
| 5a | Residual | x̂ - x̂_mse | r ∈ ℝᵈ + γ = ‖r‖ | Capture lost info |
| 5b | QJL | r ∈ ℝᵈ | sign(Sr) ∈ {±1}ᵈ | 1-bit sketch of residual |
| 5c | Correct | x̂_mse + correction | x̂_prod | Unbiased inner product |
| 6 | Rescale | x̂_prod | x̃ = ℓ · x̂_prod | Restore original magnitude |

**Theoretical guarantees:**
- MSE: $\mathbb{E}[\|x - \tilde{x}\|^2] \leq \frac{\sqrt{3\pi}}{2} \cdot \frac{1}{4^b}$
- Inner product: $\mathbb{E}[\langle q, \tilde{x}_{\text{prod}}\rangle] = \langle q, x\rangle$ (unbiased)

Both bounds are **dimension-free** — they hold for any $d$.


In [ ]:
# Final sanity check: run the full pipeline on our original vector
print("=" * 60)
print("FULL PIPELINE DEMO on our 16-dim vector")
print("=" * 60)

# Step 1: normalise
ell  = np.linalg.norm(x)
xh   = x / ell
print(f"\n[1] Normalise:  ‖x‖ = {ell:.4f}  stored as fp32")

# Step 2: rotate
y2   = Pi @ xh
print(f"[2] Rotate:     y = Pi @ x̂  |  y std = {y2.std():.4f} ≈ 1/√d = {1/np.sqrt(D):.4f}")

# Step 3: quantise (2-bit for MSE stage)
c2b  = centroids_n01[1] * (1/np.sqrt(D))   # 1-bit for MSE stage of prod variant
idx2 = np.argmin(np.abs(y2[:, None] - c2b[None, :]), axis=1)
yh2  = c2b[idx2]
print(f"[3] Quantise:   {D} coords × 1 bit = {D} bits  (centroids: {c2b.round(4)})")

# Step 4: dequantise MSE
xh_m = Pi.T @ yh2
print(f"[4] Unrotate:   x̂_mse  ‖x̂ - x̂_mse‖ = {np.linalg.norm(xh - xh_m):.4f}")

# Step 5: QJL
r2   = xh - xh_m
gam2 = np.linalg.norm(r2)
qjl2 = np.sign(S @ r2); qjl2 = np.where(qjl2==0, 1.0, qjl2)
corr2 = (np.sqrt(np.pi/2)/D) * gam2 * (S.T @ qjl2)
xh_p = xh_m + corr2
print(f"[5] QJL:        ‖r‖ = {gam2:.4f}  |  {D} sign bits  |  ‖x̂ - x̂_prod‖ = {np.linalg.norm(xh - xh_p):.4f}")

# Step 6: rescale
x_final = ell * xh_p
print(f"[6] Rescale:    x̃_prod = ℓ · x̂_prod")
print(f"\nOriginal x:    {x.round(3)}")
print(f"Recovered x̃:  {x_final.round(3)}")
print(f"\n‖x - x̃‖₂ = {np.linalg.norm(x - x_final):.5f}")
print(f"Relative error = {np.linalg.norm(x - x_final)/np.linalg.norm(x)*100:.2f}%")

bits_used = D*1 + D*1 + 32 + 32   # idx(1-bit) + qjl(1-bit) + norm + gamma
print(f"\nBits used: {bits_used} vs original {D*32}  →  {D*32/bits_used:.1f}× compression")
